# Time-Series Forecasting with TensorFlow
**Maintainer: Harpreet Singh**

A compact example using synthetic seasonal data, chronological splits, sliding windows and a persistence baseline. Replace the synthetic series with a time-ordered real dataset before making decisions.

In [ ]:
import random, platform
import numpy as np
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("Python",platform.python_version(),"TensorFlow",tf.__version__,"Keras",keras.__version__)

## 1. Synthetic seasonal data and chronological splits
Do not randomly shuffle a time series before splitting; that can leak future information into training.

In [ ]:
n=2400
t=np.arange(n,dtype=np.float32)
series=(0.7*np.sin(2*np.pi*t/24)+0.25*np.sin(2*np.pi*t/(24*7))+0.0005*t+np.random.default_rng(SEED).normal(0,0.15,n)).astype("float32")
train_end=int(n*0.70); val_end=int(n*0.85)
mean=float(series[:train_end].mean()); std=float(series[:train_end].std())
scaled=(series-mean)/(std+1e-8)
plt.figure(figsize=(12,3)); plt.plot(series); plt.axvline(train_end); plt.axvline(val_end); plt.title("Synthetic series: chronological splits"); plt.show()

In [ ]:
WINDOW=48
def make_windows(values,start,end,window=WINDOW):
    xs,ys=[],[]
    for i in range(start,end-window): xs.append(values[i:i+window]); ys.append(values[i+window])
    return np.asarray(xs,dtype="float32")[...,None],np.asarray(ys,dtype="float32")
x_train,y_train=make_windows(scaled,0,train_end)
x_val,y_val=make_windows(scaled,train_end-WINDOW,val_end)
x_test,y_test=make_windows(scaled,val_end-WINDOW,n)
print(x_train.shape,x_val.shape,x_test.shape)

## 2. Train a small neural baseline

In [ ]:
model=keras.Sequential([keras.Input((WINDOW,1)),keras.layers.Flatten(),keras.layers.Dense(64,activation="relu"),keras.layers.Dropout(0.15),keras.layers.Dense(1)])
model.compile(optimizer=keras.optimizers.Adam(1e-3),loss="mse",metrics=["mae"])
history=model.fit(x_train,y_train,validation_data=(x_val,y_val),epochs=30,batch_size=32,callbacks=[keras.callbacks.EarlyStopping(patience=4,restore_best_weights=True)],verbose=0)
forecast=model.predict(x_test,verbose=0).reshape(-1)
persistence=x_test[:,-1,0]
def mae(a,b): return float(np.mean(np.abs(a-b)))
print({"neural_test_MAE_original_units":mae(forecast*std+mean,y_test*std+mean),"persistence_test_MAE_original_units":mae(persistence*std+mean,y_test*std+mean)})

In [ ]:
steps=min(200,len(y_test))
plt.figure(figsize=(12,4)); plt.plot(y_test[:steps]*std+mean,label="actual"); plt.plot(forecast[:steps]*std+mean,label="neural forecast"); plt.plot(persistence[:steps]*std+mean,label="persistence baseline"); plt.legend(); plt.title("Held-out future window"); plt.show()

## 3. Save model and preprocessing metadata
Keep training mean and standard deviation with the model; inference must apply the same transformation.

In [ ]:
from pathlib import Path
import json
Path("artifacts").mkdir(exist_ok=True)
model.save("artifacts/time_series_forecaster.keras")
Path("artifacts/time_series_preprocessing.json").write_text(json.dumps({"window":WINDOW,"mean":mean,"std":std,"seed":SEED,"note":"Synthetic teaching demo; validate on domain data before use."},indent=2))
print("Saved model and preprocessing metadata.")